# Метрики оценки и простые бейзлайны в RecSys

Ноутбук к лекции 2 "Методы оценки рекомендательных систем. Простые бейзлайны".

**Разделение по инструментам сделано не случайно:**

- **Метрики** считаем через `replay.metrics` (библиотека RePlay, Sber AI Lab). Модуль метрик не требует PySpark — он прекрасно работает
  с обычными `pandas.DataFrame` или Python-словарями, поэтому его удобно использовать в любом ноутбуке.
- **Простые бейзлайны** (Random, Popularity, ItemKNN) реализуем через **`implicit`** — лёгкую библиотеку без Spark и JVM,
  работающую поверх `scipy.sparse`. Сами модели RePlay (`PopRec`, `QueryPopRec`, `ItemKNN`) под капотом используют PySpark,
  и поднимать Spark-сессию ради игрушечного примера — избыточно. `implicit` даёт то же самое (в том числе TF-IDF/BM25-взвешенный
  ItemKNN, как в RePlay) буквально в несколько строк и без лишней инфраструктуры.

Связка получается естественной: обучаем бейзлайны в `implicit`, конвертируем предсказания в формат, который понимает
`replay.metrics`, и сравниваем модели по всем метрикам из лекции.


## Установка библиотек

In [ ]:
# Если библиотеки ещё не установлены, раскомментируйте:
# !pip install replay-rec implicit scipy pandas numpy


In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

np.random.seed(42)


## Часть 1. Метрики оценки — `replay.metrics`

Начнём с того, как вообще пользоваться метриками, на маленьком игрушечном примере — том же, что в документации RePlay.

`replay.metrics` понимает рекомендации и ground truth в двух форматах:
- `dict`: `{user_id: [(item_id, score), ...]}` для рекомендаций и `{user_id: [item_id, ...]}` для ground truth;
- `pandas.DataFrame` / `polars.DataFrame` / `pyspark.DataFrame` с колонками `query_id`, `item_id`, `rating`.

Ниже используем словари — так нагляднее для первого знакомства.

In [ ]:
from replay.metrics import Precision, Recall, MAP, MRR, NDCG, HitRate, OfflineMetrics

# Рекомендации модели: для каждого пользователя — список (item_id, score), отсортированный по убыванию score
recommendations = {
    1: [(3, 0.6), (7, 0.5), (10, 0.4), (11, 0.3), (2, 0.2)],
    2: [(5, 0.6), (8, 0.5), (11, 0.4), (1, 0.3), (3, 0.2)],
    3: [(4, 1.0), (9, 0.5), (2, 0.1)],
}

# Ground truth: с какими айтемами пользователь реально провзаимодействовал в тесте
ground_truth = {
    1: [5, 6, 7, 8, 9, 10],
    2: [6, 7, 4, 10, 11],
    3: [1, 2, 3, 4, 5],
}


In [ ]:
# Считаем несколько метрик по отдельности для K=2
print(Precision(2)(recommendations, ground_truth))
print(Recall(2)(recommendations, ground_truth))
print(NDCG(2)(recommendations, ground_truth))


Считать метрики по одной неудобно, если их много и нужен сразу список K.
Для этого в RePlay есть `OfflineMetrics` — он считает общую часть вычислений один раз для всех метрик сразу.

In [ ]:
metrics = [
    Precision([2, 5]),
    Recall([2, 5]),
    MAP([2, 5]),
    MRR([2, 5]),
    NDCG([2, 5]),
    HitRate([2, 5]),
]

result = OfflineMetrics(metrics)(recommendations, ground_truth)
pd.Series(result).sort_index()


### Beyond-accuracy метрики

Для `Coverage`, `Novelty`, `Surprisal` дополнительно нужен `train` — с чем пользователи взаимодействовали при обучении
(чтобы понять, какие рекомендации *новые*, и какая доля каталога вообще попадает в рекомендации).

In [ ]:
from replay.metrics import Coverage, Novelty, Surprisal

train = {
    1: [5, 6, 8, 9, 2],
    2: [5, 8, 11, 1, 3],
    3: [4, 9, 2],
}

beyond_accuracy = [Coverage(2), Novelty(2), Surprisal(2)]
result = OfflineMetrics(beyond_accuracy)(recommendations, ground_truth, train)
pd.Series(result).sort_index()


### Сравнение нескольких моделей — `Experiment`

На практике интересен не отдельный прогон, а сравнение нескольких моделей между собой.
`Experiment` хранит результаты по каждой модели и умеет показывать разницу в процентах относительно baseline.

In [ ]:
from replay.metrics import Experiment

# Вторая, чуть более удачная модель — для примера
recommendations_v2 = {
    1: [(7, 0.9), (10, 0.8), (6, 0.7), (5, 0.6), (2, 0.2)],
    2: [(7, 0.8), (6, 0.7), (10, 0.6), (11, 0.5), (3, 0.2)],
    3: [(4, 1.0), (2, 0.6), (5, 0.5)],
}

exp = Experiment([NDCG([2, 5]), Recall([2, 5]), Coverage(5)], ground_truth, train)
exp.add_result("baseline", recommendations)
exp.add_result("model_v2", recommendations_v2)

exp.results


In [ ]:
exp.compare("baseline")


## Часть 2. Простые бейзлайны — вручную + `implicit`

Для демонстрации соберём небольшой синтетический датасет: у пользователей и айтемов есть скрытые векторы предпочтений,
поверх которых сгенерируем implicit-взаимодействия (клики), плюс сделаем часть айтемов заметно более популярными —
это нужно, чтобы Popularity-бейзлайн был содержательным, а не случайным.

In [ ]:
n_users = 300
n_items = 150
latent_dim = 8

user_vecs = np.random.normal(size=(n_users, latent_dim))
item_vecs = np.random.normal(size=(n_items, latent_dim))

# Добавляем перекос по популярности (несколько айтемов должны быть очень популярны — длинный хвост)
item_popularity_bias = np.random.pareto(a=2.0, size=n_items)

scores = user_vecs @ item_vecs.T + item_popularity_bias[None, :]
probs = 1 / (1 + np.exp(-scores / 3))

interactions = (np.random.rand(n_users, n_items) < probs).astype(np.float32)

print("Средняя плотность матрицы:", interactions.mean())
print("Взаимодействий на пользователя (в среднем):", interactions.sum(axis=1).mean())


### Train/test split

Для каждого пользователя случайно отложим ~20% его взаимодействий в тест (как в лекции — user-based holdout).

In [ ]:
train_mask = np.ones_like(interactions)
test_items = {}

for u in range(n_users):
    liked = np.where(interactions[u] > 0)[0]
    if len(liked) < 2:
        continue
    n_test = max(1, int(len(liked) * 0.2))
    test_idx = np.random.choice(liked, size=n_test, replace=False)
    test_items[u] = test_idx.tolist()
    train_mask[u, test_idx] = 0

train_matrix = interactions * train_mask
train_csr = csr_matrix(train_matrix)

# Ground truth в формате replay.metrics: {user_id: [item_id, ...]}
ground_truth_gt = {u: items for u, items in test_items.items()}
train_gt = {u: np.where(train_matrix[u] > 0)[0].tolist() for u in range(n_users)}

print("Пользователей с тестовыми айтемами:", len(ground_truth_gt))


### Бейзлайн 1 — Random

Самый простой бейзлайн не нуждается ни в какой библиотеке — это несколько строк на numpy.

In [ ]:
def random_recommend(n_users, n_items, k=10, seed=0):
    rng = np.random.default_rng(seed)
    recs = {}
    for u in range(n_users):
        items = rng.choice(n_items, size=k, replace=False)
        scores = np.linspace(1.0, 0.1, k)
        recs[u] = list(zip(items.tolist(), scores.tolist()))
    return recs

random_recs = random_recommend(n_users, n_items, k=10)


### Бейзлайн 2 — Most Popular

Тоже не требует библиотек: считаем популярность по train и отдаём всем один и тот же топ,
за вычетом айтемов, которые пользователь уже видел в train.

In [ ]:
def popularity_recommend(train_matrix, k=10):
    item_popularity = train_matrix.sum(axis=0)
    order = np.argsort(-item_popularity)

    recs = {}
    for u in range(train_matrix.shape[0]):
        seen = set(np.where(train_matrix[u] > 0)[0])
        rec_items = [i for i in order if i not in seen][:k]
        scores = item_popularity[rec_items]
        recs[u] = list(zip(rec_items, scores.tolist()))
    return recs

popularity_recs = popularity_recommend(train_matrix, k=10)


### Бейзлайн 3 — ItemKNN через `implicit`

А вот здесь без библиотеки уже не так удобно: нужно считать похожесть между айтемами и агрегировать её по истории
пользователя. `implicit.nearest_neighbours` даёт готовый ItemKNN с разными взвешиваниями — `CosineRecommender`
(как в RePlay без взвешивания), `TFIDFRecommender` и `BM25Recommender` (как параметр `weighting` в `replay.models.ItemKNN`).

In [ ]:
from implicit.nearest_neighbours import CosineRecommender, BM25Recommender

def fit_itemknn(model_cls, train_csr, k_neighbours=20, **kwargs):
    model = model_cls(K=k_neighbours, **kwargs)
    model.fit(train_csr)
    return model

def itemknn_recommend(model, train_csr, k=10):
    user_ids = np.arange(train_csr.shape[0])
    ids, scores = model.recommend(
        user_ids, train_csr[user_ids], N=k, filter_already_liked_items=True
    )
    recs = {}
    for row, u in enumerate(user_ids):
        recs[u] = list(zip(ids[row].tolist(), scores[row].tolist()))
    return recs

knn_cosine = fit_itemknn(CosineRecommender, train_csr, k_neighbours=20)
knn_cosine_recs = itemknn_recommend(knn_cosine, train_csr, k=10)

knn_bm25 = fit_itemknn(BM25Recommender, train_csr, k_neighbours=20)
knn_bm25_recs = itemknn_recommend(knn_bm25, train_csr, k=10)


## Часть 3. Сводим всё вместе — сравниваем бейзлайны через RePlay

Теперь, когда у нас есть предсказания четырёх бейзлайнов в едином формате `{user_id: [(item_id, score), ...]}`,
можно прогнать их через тот же `OfflineMetrics`, которым мы пользовались в части 1.

In [ ]:
from replay.metrics import Precision, Recall, MAP, NDCG, Coverage, Novelty

metrics = [
    Precision(10),
    Recall(10),
    MAP(10),
    NDCG(10),
    Coverage(10),
    Novelty(10),
]

all_recs = {
    "Random": random_recs,
    "Most Popular": popularity_recs,
    "ItemKNN (cosine)": knn_cosine_recs,
    "ItemKNN (BM25)": knn_bm25_recs,
}

rows = {}
for name, recs in all_recs.items():
    rows[name] = OfflineMetrics(metrics)(recs, ground_truth_gt, train_gt)

pd.DataFrame(rows).T


### Что должно получиться

- **Random** — почти нулевая точность, но максимальный `Coverage` и `Novelty`.
- **Most Popular** — заметно точнее случайного, но `Coverage` низкий: рекомендует всем одно и то же.
- **ItemKNN** — обычно точнее Popularity за счёт персонализации, при этом `Coverage` выше, чем у Popularity,
  так как разным пользователям рекомендуются разные айтемы. Разница между `cosine` и `BM25` взвешиванием
  показывает, насколько чувствителен ItemKNN к перевзвешиванию популярных айтемов — ровно то, о чём говорили в лекции.

То же самое можно оформить через `Experiment`, чтобы сразу увидеть относительный прирост каждой модели
над случайным бейзлайном:

In [ ]:
exp = Experiment(metrics, ground_truth_gt, train_gt)
for name, recs in all_recs.items():
    exp.add_result(name, recs)

exp.compare("Random")


## Итоги

- Метрики (accuracy и beyond-accuracy) удобно считать через `replay.metrics` — модуль не завязан на Spark
  и одинаково хорошо работает что с игрушечными словарями, что с реальными `DataFrame`.
- Простые бейзлайны эффективнее прототипировать без тяжёлых зависимостей: Random и Popularity — это numpy
  в 5-10 строк, а для ItemKNN отлично подходит `implicit` — лёгкая библиотека с готовыми cosine/TF-IDF/BM25
  реализациями, которые концептуально совпадают с параметрами `ItemKNN` из RePlay.
- Полноценные модели самого RePlay (`PopRec`, `QueryPopRec`, `ItemKNN` на PySpark) имеет смысл подключать,
  когда датасет уже не помещается в память одной машины и нужна распределённая обработка — то есть на
  следующем этапе исследования, а не на этапе быстрого прототипирования бейзлайнов.
